# 04 · Visualización y predicciones

Objetivo: cargar los modelos guardados, generar predicciones nuevas sobre la partición de validación y producir las figuras solicitadas sin volver a entrenar.

Proyecto de ECG con dos etapas: reposo/actividad y clasificación de cuatro ritmos. Ejecutar los cuadernos **01 → 02 → 03 → 04** con la misma carpeta de resultados. Estos archivos se entregan sin salidas ejecutadas: no contienen métricas inventadas.


## Preparación en Colab
Instalar dependencias, montar Drive y editar las rutas. Subir previamente los datos a Drive; no hace falta subir el ZIP de 2,5 GB si se usa el Parquet Chapman ya consolidado. Para entrenar ResNet, seleccionar un entorno con GPU. El código también admite CPU, con mayor tiempo de ejecución.

Referencias: [entrada y salida de archivos en Colab](https://colab.research.google.com/notebooks/io.ipynb) y [preguntas frecuentes](https://research.google.com/colaboratory/faq.html).


In [ ]:
%pip -q install pyarrow seaborn joblib scipy scikit-learn skl2onnx onnx


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# EDITAR: carpeta del proyecto en tu Google Drive. Usar la misma en los 4 cuadernos.
PROJECT_DIR = Path('/content/drive/MyDrive/TP_IA')
REST_DIR = PROJECT_DIR / 'RestingClass' / 'data_resting'
ACTIVE_DIR = PROJECT_DIR / 'RestingClass' / 'data_not_resting'
CHAPMAN_FILE = PROJECT_DIR / 'Detector' / 'chapman_lead_I_consolidated.parquet'
STAGES = [1, 2]  # Usar [2] si primero quieres trabajar solamente con Chapman.
RESULTS = PROJECT_DIR / 'results_colab'
PREPARED = RESULTS / 'prepared'
MODELS = RESULTS / 'models'
FIGURES = RESULTS / 'figures'
for directory in (RESULTS, PREPARED, MODELS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
assert STAGES and set(STAGES) <= {1, 2}, 'Elegir etapas 1 y/o 2.'
SEED = 42
print('Resultados:', RESULTS)


In [ ]:
import json, random, uuid, hashlib, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, classification_report
from IPython.display import display

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {1: ['Reposo', 'Actividad'], 2: ['Bradicardia sinusal', 'Taquicardia sinusal', 'Fibrilación auricular', 'Ritmo sinusal normal']}
FEATURE_NAMES = {
    1: ['Kurtosis', 'ZCR', 'Crest_Factor', 'Skewness', 'HR', 'RMSSD', 'SDNN', 'EMG_Power', 'Baseline_Power', 'LF_HF_Ratio', 'Morph_Corr', 'Std_Dev'],
    2: ['hr', 'sdnn', 'rmssd', 'pnn50', 'pnn20', 'rr_cv', 'sd1', 'sd2', 'sd1_sd2', 'skew', 'kurt']
}
def save_json(filename, value):
    Path(filename).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def load_prepared(stage):
    filename = PREPARED / f'stage{stage}.npz'
    if not filename.exists():
        raise FileNotFoundError(f'{filename}: ejecutar primero el cuaderno 01 para esta etapa.')
    with np.load(filename, allow_pickle=False) as stored:
        data = {key: stored[key] for key in stored.files}
    return data

def metrics(y, pred, probs):
    # Macro para comparar con igual peso todas las clases, en ambas etapas.
    auc = roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2 else roc_auc_score(y, probs, multi_class='ovr', average='macro')
    return dict(accuracy=accuracy_score(y, pred), precision_macro=precision_score(y, pred, average='macro', zero_division=0), recall_macro=recall_score(y, pred, average='macro', zero_division=0), f1_macro=f1_score(y, pred, average='macro', zero_division=0), auc=auc)

def make_cv(stage, y, groups, folds=3):
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=SEED) if stage == 1 else StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    splits = list(cv.split(np.zeros((len(y), 1)), y, groups if stage == 1 else None))
    for tr, va in splits:
        if len(np.unique(y[tr])) != len(CLASS_NAMES[stage]) or len(np.unique(y[va])) != len(CLASS_NAMES[stage]):
            raise ValueError('Un fold no contiene todas las clases. Revisar sujetos/clases o reducir folds.')
        if stage == 1:
            assert not set(groups[tr]) & set(groups[va]), 'Sujetos compartidos entre folds.'
    return splits

def make_loader(signals, labels, batch_size, shuffle=False):
    # Normalización por señal; no ajusta estadísticas sobre otras muestras.
    a = np.asarray(signals, dtype=np.float32)
    a = (a - a.mean(axis=1, keepdims=True)) / (a.std(axis=1, keepdims=True) + 1e-8)
    dataset = TensorDataset(torch.from_numpy(a).unsqueeze(1), torch.as_tensor(labels, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


class BasicBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 5, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))

class ResNet1D(nn.Module):
    def __init__(self, classes, width, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, width, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(width)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock1D(width, width), BasicBlock1D(width, width))
        self.layer2 = nn.Sequential(BasicBlock1D(width, width*2, 2), BasicBlock1D(width*2, width*2))
        self.layer3 = nn.Sequential(BasicBlock1D(width*2, width*4, 2), BasicBlock1D(width*4, width*4))
        self.layer4 = nn.Sequential(BasicBlock1D(width*4, width*8, 2), BasicBlock1D(width*8, width*8))
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(width*8, classes))
    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.avgpool(x).flatten(1))

def predict_network(model, loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for signals, _ in loader:
            chunks.append(F.softmax(model(signals.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(chunks)

print('Dispositivo:', DEVICE)
versions = {name: __import__(name).__version__ for name in ['numpy', 'pandas', 'sklearn', 'scipy', 'torch', 'joblib']}
versions['python'] = platform.python_version()
save_json(RESULTS / 'environment.json', versions)
display(versions)


## Inferencia reproducible y tabla de ejemplos
Se verifica que los modelos pertenezcan a la versión de datos preparada en 01. La confianza es la probabilidad máxima; no equivale a certeza clínica. Se muestran datos reales reservados de validación, no ejemplos seleccionados para mejorar artificialmente las métricas.


In [ ]:
all_metrics = []
for stage in STAGES:
    data = load_prepared(stage)
    tr, va = data['train_idx'], data['val_idx']
    y = data['y'][va]
    predictions = {}
    for name in ('baseline', 'baseline_rf', 'baseline_resnet', 'rf', 'lr', 'resnet'):
        meta_file = MODELS / f'stage{stage}_{name}_meta.json'
        if not meta_file.exists():
            print(f'Etapa {stage}: {name} omitido, no hay metadatos de entrenamiento.')
            continue
        meta = json.loads(meta_file.read_text())
        assert meta['dataset_id'] == str(data['dataset_id']), f'{name}: datos cambiaron; repetir entrenamiento.'
        if name in ('resnet', 'baseline_resnet'):
            model = ResNet1D(**meta['config']).to(DEVICE)
            model.load_state_dict(torch.load(MODELS / f'stage{stage}_{name}.pth', map_location=DEVICE, weights_only=True))
            probs = predict_network(model, make_loader(data['raw'][va], y, 128))
        else:
            model = joblib.load(MODELS / f'stage{stage}_{name}.joblib')
            probs = model.predict_proba(data['features'][va])
        pred = probs.argmax(axis=1)
        predictions[name] = (pred, probs)
        all_metrics.append(dict(stage=stage, model=name, **metrics(y, pred, probs)))
        names = np.asarray(CLASS_NAMES[stage])
        table = pd.DataFrame(dict(id=data['ids'][va], real=names[y], predicted=names[pred], correct=pred == y, confidence=probs.max(axis=1), error_score=1-probs[np.arange(len(y)), y]))
        for index, label in enumerate(CLASS_NAMES[stage]):
            table[f'p_{label}'] = probs[:, index]
        table.to_csv(RESULTS / f'stage{stage}_{name}_predictions.csv', index=False)
        print(f'Etapa {stage} / {name}')
        display(table.head(10))
        print(classification_report(y, pred, target_names=CLASS_NAMES[stage], zero_division=0))
        # Matriz de confusión + histograma de error probabilístico.
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        ConfusionMatrixDisplay.from_predictions(y, pred, labels=np.arange(len(names)), display_labels=names, ax=axes[0], cmap='Blues', colorbar=False, xticks_rotation=35)
        axes[0].set_title(f'Etapa {stage}: {name}')
        error = table['error_score'].to_numpy()
        for correct, label in [(True, 'Aciertos'), (False, 'Errores')]:
            subset = error[(pred == y) == correct]
            if len(subset):
                axes[1].hist(subset, bins=np.linspace(0,1,21), alpha=0.6, label=label)
        axes[1].set(xlabel='1 − probabilidad asignada a la clase real', ylabel='Cantidad de ejemplos', title='Distribución del error probabilístico')
        axes[1].legend()
        fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_{name}_errors.png', dpi=150); plt.show(); plt.close(fig)
    # Ejemplos de señales con predicción, priorizando errores para analizarlos.
    if not predictions:
        raise FileNotFoundError('No hay modelos: ejecutar 02 y 03.')
    name = 'resnet' if 'resnet' in predictions else 'rf' if 'rf' in predictions else next(iter(predictions))
    pred, probs = predictions[name]
    selected = np.concatenate([np.flatnonzero(pred != y), np.flatnonzero(pred == y)])[:4]
    fig, axes = plt.subplots(len(selected), 1, figsize=(12, 2.5*len(selected)), squeeze=False)
    for ax, index in zip(axes.ravel(), selected):
        signal = data['raw'][va[index]]
        ax.plot(np.arange(len(signal))/500, signal, lw=0.7)
        ax.set_title(f'{name} | real: {CLASS_NAMES[stage][y[index]]} | predicción: {CLASS_NAMES[stage][pred[index]]} | confianza: {probs[index].max():.3f}')
        ax.set(xlabel='Tiempo (s)', ylabel='Amplitud')
    fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_prediction_examples.png', dpi=150); plt.show(); plt.close(fig)
    del data
    gc.collect()
summary = pd.DataFrame(all_metrics)
summary.to_csv(RESULTS / 'validation_metrics.csv', index=False)
display(summary)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=summary, x='model', y='f1_macro', hue='stage', ax=ax, errorbar=None)
ax.set(ylim=(0,1), ylabel='F1 macro de validación', title='Comparación de modelos')
fig.tight_layout(); fig.savefig(FIGURES / 'model_comparison.png', dpi=150); plt.show(); plt.close(fig)


## Convergencia de ResNet
La pérdida y F1 macro se trazan por época. La línea vertical marca el checkpoint seleccionado. Las métricas de entrenamiento se acumulan durante las actualizaciones de pesos, mientras las de validación usan los pesos al final de la época. Interpretar separaciones y oscilaciones sin atribuir una causa única.


In [ ]:
for stage in STAGES:
    history_file = RESULTS / f'stage{stage}_resnet_history.csv'
    meta_file = MODELS / f'stage{stage}_resnet_meta.json'
    if not history_file.exists() or not meta_file.exists():
        print(f'Sin historial ResNet etapa {stage}; activar TRAIN_RESNET en 03.'); continue
    meta = json.loads(meta_file.read_text())
    with np.load(PREPARED / f'stage{stage}.npz', allow_pickle=False) as stored:
        assert meta['dataset_id'] == str(stored['dataset_id']), 'Historial de otra versión de datos'
    history = pd.read_csv(history_file)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, column, title in [(axes[0], 'loss', 'Pérdida'), (axes[1], 'f1_macro', 'F1 macro')]:
        ax.plot(history['epoch'], history[f'train_{column}'], label='Entrenamiento')
        ax.plot(history['epoch'], history[f'val_{column}'], label='Validación')
        ax.axvline(meta['best_epoch'], linestyle='--', color='gray', label='Mejor checkpoint')
        ax.set(xlabel='Época', ylabel=title, title=f'Etapa {stage}: {title}')
        ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_convergence.png', dpi=150); plt.show(); plt.close(fig)


## Importancia de características y curvas de aprendizaje
Las curvas clásicas muestran desempeño frente al número de ejemplos, no convergencia por épocas. Son opcionales porque vuelven a ajustar modelos dentro de validación cruzada. El escalador permanece dentro del pipeline.


In [ ]:
GENERATE_LEARNING_CURVES = False  # Activar si se dispone de tiempo.
for stage in STAGES:
    data = load_prepared(stage)
    tr = data['train_idx']
    rf = joblib.load(MODELS / f'stage{stage}_rf.joblib')
    meta = json.loads((MODELS / f'stage{stage}_rf_meta.json').read_text())
    assert meta['dataset_id'] == str(data['dataset_id'])
    importance = pd.DataFrame({'feature': FEATURE_NAMES[stage], 'importance': rf.named_steps['rf'].feature_importances_}).sort_values('importance')
    importance.to_csv(RESULTS / f'stage{stage}_feature_importance.csv', index=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance['feature'], importance['importance'])
    ax.set(title=f'Etapa {stage}: importancia Random Forest', xlabel='Importancia por reducción de impureza')
    fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_feature_importance.png', dpi=150); plt.show(); plt.close(fig)
    if GENERATE_LEARNING_CURVES:
        cv = make_cv(stage, data['y'][tr], data['groups'][tr])
        sizes, training, validation = learning_curve(rf, data['features'][tr], data['y'][tr], cv=cv, train_sizes=np.linspace(0.3, 1.0, 5), scoring='f1_macro', n_jobs=2, error_score=np.nan)
        fig, ax = plt.subplots(figsize=(7, 4))
        for values, label in [(training, 'Entrenamiento'), (validation, 'Validación cruzada')]:
            mean, std = np.nanmean(values, axis=1), np.nanstd(values, axis=1)
            ax.plot(sizes, mean, 'o-', label=label)
            ax.fill_between(sizes, mean-std, mean+std, alpha=0.15)
        ax.set(xlabel='Ejemplos de entrenamiento', ylabel='F1 macro', title=f'Etapa {stage}: curva de aprendizaje RF')
        ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_learning_curve.png', dpi=150); plt.show(); plt.close(fig)
        if np.isnan(validation).any():
            print('Hay folds/tamaños sin resultados: revisar soporte por clase y avisos antes de interpretar.')
    del data


## Conclusiones para completar con los resultados ejecutados
1. ¿Qué modelo obtuvo mayor F1 macro y qué clases presentan más confusiones?
2. ¿La optimización mejoró respecto del modelo base? Comparar sobre la misma partición.
3. ¿Qué muestran la pérdida y el F1 de validación acerca de estabilidad y generalización?
4. ¿Qué errores ocurren con mayor confianza y qué señales los explican?
5. Limitaciones: datos de validación usados en selección de checkpoint, frecuencia asumida, extracción de picos, ventanas cortas y posible dependencia por paciente no verificada en Chapman.
6. La prueba dinámica en Vercel demuestra funcionamiento de la aplicación; documentar sus ejemplos. Solo puede considerarse evaluación independiente si se acredita que sus señales no participaron en entrenamiento ni ajuste.


## Descargar resultados
Las salidas ya están guardadas en Drive. La siguiente celda permite descargar tablas, modelos y figuras en un ZIP; excluye las matrices de señales preparadas para evitar duplicar cientos de MB.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/resultados_tp_ia.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for filename in RESULTS.rglob('*'):
        if filename.is_file() and filename.suffix != '.npz':
            z.write(filename, filename.relative_to(RESULTS))
files.download(str(archive))
